In [3]:
# Task 1 - Import Libraries

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, brier_score_loss

In [4]:
# Task 1 - Load Feature Data

df = pd.read_csv("afl_feature_table_v1.csv")

df.head()

,team_name,opponent,match_date,venue,win_streak,last_5_avg_score,head_to_head_wins
0,\t Adelaide Crows,hawthorn hawks,1991-03-22,AAMI Stadium,0.0,NaN,0
1,\t Adelaide Crows,North Melbourne Kangaroos,1991-09-01,AAMI Stadium,1.0,NaN,1
2,\t Adelaide Crows,West Coast Eagles,1993-04-04,AAMI Stadium,2.0,NaN,0
3,\t Adelaide Crows,Melbourne Demons,1993-04-25,Melbourne Cricket Ground,3.0,NaN,0
4,\t Adelaide Crows,North Melbourne Kangaroos,1993-05-16,AAMI Stadium,0.0,NaN,2


In [5]:
# Task 1 - Check Target

print(df["match_winner"].value_counts())

KeyError: 'match_winner'

In [6]:
# Task 1 - Check Available Columns

print(df.columns.tolist())

['team_name', 'opponent', 'match_date', 'venue', 'win_streak', 'last_5_avg_score', 'head_to_head_wins']


In [7]:
# Task 1 - Check Original Data

print(home_matches.columns.tolist())

NameError: name 'home_matches' is not defined

In [8]:
# Task 1 - Check Number of Rows

print("Feature table rows:", len(df))

Feature table rows: 15808


In [9]:
# Task 1 - Load Team Matches

team_matches = pd.read_csv("team_matches_home_away_raw - team_matches_home_away_raw.csv.csv")

print("Team Matches rows:", len(team_matches))

Team Matches rows: 15808


In [10]:
# Task 1 - Create Match Winner Target

home_matches = team_matches[team_matches["home_away"] == "H"].copy()

home_matches["match_winner"] = home_matches["result"].replace({
    "W": "Home Win",
    "L": "Away Win",
    "D": "Draw"
})

print(home_matches["match_winner"].value_counts())

match_winner
Home Win    4669
Away Win    3170
Draw          65
Name: count, dtype: int64


In [11]:
# Task 1 - Check Feature Data Rows

print(df[["team_name", "opponent", "match_date"]].head())
print()
print(team_matches[["team_name", "opponent", "match_date"]].head())

            team_name                   opponent  match_date
0  \t Adelaide Crows              hawthorn hawks  1991-03-22
1  \t Adelaide Crows   North Melbourne Kangaroos  1991-09-01
2  \t Adelaide Crows           West Coast Eagles  1993-04-04
3  \t Adelaide Crows            Melbourne Demons  1993-04-25
4  \t Adelaide Crows   North Melbourne Kangaroos  1993-05-16

                   team_name                   opponent  match_date
0             Hawthorn Hawks  North Melbourne Kangaroos  1994-09-10
1  North Melbourne Kangaroos             Hawthorn Hawks  1994-09-10
2  North Melbourne Kangaroos             Brisbane Lions  2008-05-31
3               Sydney Swans           Melbourne Demons  2017-06-30
4               Sydney Swans               Geelong Cats  2019-06-01


In [ ]:
# Task 1 - Load Feature Data

df = pd.read_csv("afl_feature_table_v1.csv")

df.head()

,team_name,opponent,match_date,venue,win_streak,last_5_avg_score,head_to_head_wins
0,\t Adelaide Crows,hawthorn hawks,1991-03-22,AAMI Stadium,0.0,NaN,0
1,\t Adelaide Crows,North Melbourne Kangaroos,1991-09-01,AAMI Stadium,1.0,NaN,1
2,\t Adelaide Crows,West Coast Eagles,1993-04-04,AAMI Stadium,2.0,NaN,0
3,\t Adelaide Crows,Melbourne Demons,1993-04-25,Melbourne Cricket Ground,3.0,NaN,0
4,\t Adelaide Crows,North Melbourne Kangaroos,1993-05-16,AAMI Stadium,0.0,NaN,2


In [ ]:
# Task 1 - Check Target

print(df["match_winner"].value_counts())

KeyError: 'match_winner'

In [12]:
# Task 1 - Clean Team Names

df["team_name"] = df["team_name"].str.strip().str.lower()
df["opponent"] = df["opponent"].str.strip().str.lower()

team_matches["team_name"] = team_matches["team_name"].str.strip().str.lower()
team_matches["opponent"] = team_matches["opponent"].str.strip().str.lower()

print(df[["team_name", "opponent", "match_date"]].head())
print()
print(team_matches[["team_name", "opponent", "match_date"]].head())

        team_name                   opponent  match_date
0  adelaide crows             hawthorn hawks  1991-03-22
1  adelaide crows  north melbourne kangaroos  1991-09-01
2  adelaide crows          west coast eagles  1993-04-04
3  adelaide crows           melbourne demons  1993-04-25
4  adelaide crows  north melbourne kangaroos  1993-05-16

                   team_name                   opponent  match_date
0             hawthorn hawks  north melbourne kangaroos  1994-09-10
1  north melbourne kangaroos             hawthorn hawks  1994-09-10
2  north melbourne kangaroos             brisbane lions  2008-05-31
3               sydney swans           melbourne demons  2017-06-30
4               sydney swans               geelong cats  2019-06-01


In [13]:
# Task 1 - Check Matching Rows

matched = df.merge(
    team_matches[["team_name", "opponent", "match_date", "result"]],
    on=["team_name", "opponent", "match_date"],
    how="left"
)

print("Total feature rows:", len(df))
print("Rows with result:", matched["result"].notna().sum())
print("Rows without result:", matched["result"].isna().sum())

Total feature rows: 15808
Rows with result: 15808
Rows without result: 0


In [14]:
# Task 1 - Create Match Winner Target

matched["match_winner"] = np.where(
    matched["home_away"] == "H",
    matched["result"].map({
        "W": "Home Win",
        "L": "Away Win",
        "D": "Draw"
    }),
    matched["result"].map({
        "W": "Away Win",
        "L": "Home Win",
        "D": "Draw"
    })
)

print(matched["match_winner"].value_counts())

KeyError: 'home_away'

In [15]:
# Task 1 - Add Home/Away Information

matched = df.merge(
    team_matches[["team_name", "opponent", "match_date", "result", "home_away"]],
    on=["team_name", "opponent", "match_date"],
    how="left"
)

print(matched[["team_name", "opponent", "home_away", "result"]].head())

        team_name                   opponent home_away result
0  adelaide crows             hawthorn hawks         H      W
1  adelaide crows  north melbourne kangaroos         H      W
2  adelaide crows          west coast eagles         H      W
3  adelaide crows           melbourne demons         A      L
4  adelaide crows  north melbourne kangaroos         H      W


In [ ]:
# Task 1 - Check Target

print(df["match_winner"].value_counts())

KeyError: 'match_winner'

In [16]:
# Task 1 - Create Match Winner

matched["match_winner"] = np.where(
    matched["home_away"] == "H",
    matched["result"].map({
        "W": "Home Win",
        "L": "Away Win",
        "D": "Draw"
    }),
    matched["result"].map({
        "W": "Away Win",
        "L": "Home Win",
        "D": "Draw"
    })
)

print(matched["match_winner"].value_counts())

match_winner
Home Win    9338
Away Win    6340
Draw         130
Name: count, dtype: int64


In [17]:
# Task 1 - Keep Home Team Rows

match_data = matched[matched["home_away"] == "H"].copy()

print("Match rows:", len(match_data))
print(match_data["match_winner"].value_counts())

Match rows: 7904
match_winner
Home Win    4669
Away Win    3170
Draw          65
Name: count, dtype: int64


In [18]:
# Task 1 - Match Winner Baseline

baseline_prediction = "Home Win"

match_data["baseline_prediction"] = baseline_prediction

baseline_accuracy = accuracy_score(
    match_data["match_winner"],
    match_data["baseline_prediction"]
)

print("Baseline Accuracy:", baseline_accuracy)

Baseline Accuracy: 0.5907135627530364


In [19]:
# Task 1 - Baseline F1 Score

baseline_f1 = f1_score(
    match_data["match_winner"],
    match_data["baseline_prediction"],
    average="weighted"
)

print("Baseline F1 Score:", baseline_f1)

Baseline F1 Score: 0.43872450878770813


In [20]:
# Task 1 - Import Libraries

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, brier_score_loss

In [ ]:
# Task 1 - Load Feature Data

df = pd.read_csv("afl_feature_table_v1.csv")

df.head()

,team_name,opponent,match_date,venue,win_streak,last_5_avg_score,head_to_head_wins
0,\t Adelaide Crows,hawthorn hawks,1991-03-22,AAMI Stadium,0.0,NaN,0
1,\t Adelaide Crows,North Melbourne Kangaroos,1991-09-01,AAMI Stadium,1.0,NaN,1
2,\t Adelaide Crows,West Coast Eagles,1993-04-04,AAMI Stadium,2.0,NaN,0
3,\t Adelaide Crows,Melbourne Demons,1993-04-25,Melbourne Cricket Ground,3.0,NaN,0
4,\t Adelaide Crows,North Melbourne Kangaroos,1993-05-16,AAMI Stadium,0.0,NaN,2


In [ ]:
# Task 1 - Check Target

print(df["match_winner"].value_counts())

KeyError: 'match_winner'

In [23]:
# Task 1 - Match Winner Baseline

baseline_prediction = "Home Win"

match_data["baseline_prediction"] = baseline_prediction

baseline_accuracy = accuracy_score(
    match_data["match_winner"],
    match_data["baseline_prediction"]
)

print("Baseline Accuracy:", baseline_accuracy)

Baseline Accuracy: 0.5907135627530364


In [24]:
# Task 1 - Baseline F1 Score

baseline_f1 = f1_score(
    match_data["match_winner"],
    match_data["baseline_prediction"],
    average="weighted"
)

print("Baseline F1 Score:", baseline_f1)

Baseline F1 Score: 0.43872450878770813


In [25]:
# Task 1 - Baseline ROC AUC

baseline_roc_auc = np.nan

print("Baseline ROC AUC:", baseline_roc_auc)

Baseline ROC AUC: nan


In [26]:
# Task 1 - Baseline Results

baseline_results = pd.DataFrame({
    "Metric": ["Accuracy", "F1 Score", "ROC AUC"],
    "Baseline": [
        baseline_accuracy,
        baseline_f1,
        baseline_roc_auc
    ]
})

baseline_results

,Metric,Baseline
0,Accuracy,0.590714
1,F1 Score,0.438725
2,ROC AUC,NaN


In [ ]:
# Task 1 - Load Feature Data

df = pd.read_csv("afl_feature_table_v1.csv")

df.head()

,team_name,opponent,match_date,venue,win_streak,last_5_avg_score,head_to_head_wins
0,\t Adelaide Crows,hawthorn hawks,1991-03-22,AAMI Stadium,0.0,NaN,0
1,\t Adelaide Crows,North Melbourne Kangaroos,1991-09-01,AAMI Stadium,1.0,NaN,1
2,\t Adelaide Crows,West Coast Eagles,1993-04-04,AAMI Stadium,2.0,NaN,0
3,\t Adelaide Crows,Melbourne Demons,1993-04-25,Melbourne Cricket Ground,3.0,NaN,0
4,\t Adelaide Crows,North Melbourne Kangaroos,1993-05-16,AAMI Stadium,0.0,NaN,2


In [ ]:
# Task 1 - Check Target

print(df["match_winner"].value_counts())

KeyError: 'match_winner'

In [27]:
# Task 2 - Prepare Features and Target

X = match_data[
    [
        "team_name",
        "opponent",
        "venue",
        "win_streak",
        "last_5_avg_score",
        "head_to_head_wins"
    ]
]

y = match_data["match_winner"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (7904, 6)
Target shape: (7904,)


In [28]:
# Task 2 - Split Data into Training and Testing

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Training rows: 6323
Testing rows: 1581


In [29]:
# Task 2 - Create Preprocessing

categorical_features = [
    "team_name",
    "opponent",
    "venue"
]

numerical_features = [
    "win_streak",
    "last_5_avg_score",
    "head_to_head_wins"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", StandardScaler(), numerical_features)
    ]
)

print("Preprocessing is ready")

Preprocessing is ready


In [30]:
# Task 2 - Logistic Regression Model

logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=1000))
    ]
)

logistic_model.fit(X_train, y_train)

print("Logistic Regression model trained")

ValueError: Input X contains NaN.
LogisticRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [31]:
# Task 2 - Import Missing Value Handler

from sklearn.impute import SimpleImputer

In [33]:
# Task 2 - Update Preprocessing for Missing Values

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

numerical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", categorical_pipeline, categorical_features),
        ("num", numerical_pipeline, numerical_features)
    ]
)

print("Updated preprocessing is ready")

Updated preprocessing is ready


In [34]:
# Task 2 - Train Logistic Regression

logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=1000))
    ]
)

logistic_model.fit(X_train, y_train)

print("Logistic Regression model trained successfully")

Logistic Regression model trained successfully


In [35]:
# Task 2 - Logistic Regression Predictions

y_pred_logistic = logistic_model.predict(X_test)

y_prob_logistic = logistic_model.predict_proba(X_test)

print("Predictions completed")

Predictions completed


In [36]:
# Task 2 - Logistic Regression Evaluation

logistic_accuracy = accuracy_score(y_test, y_pred_logistic)

logistic_f1 = f1_score(
    y_test,
    y_pred_logistic,
    average="weighted"
)

logistic_roc_auc = roc_auc_score(
    y_test,
    y_prob_logistic,
    multi_class="ovr"
)

print("Logistic Regression Accuracy:", logistic_accuracy)
print("Logistic Regression F1 Score:", logistic_f1)
print("Logistic Regression ROC AUC:", logistic_roc_auc)

Logistic Regression Accuracy: 0.6065781151170145
Logistic Regression F1 Score: 0.5810526763135818
Logistic Regression ROC AUC: 0.6135028415721717


In [37]:
# Task 2 - Random Forest Model

random_forest_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=100,
            random_state=42
        ))
    ]
)

random_forest_model.fit(X_train, y_train)

print("Random Forest model trained successfully")

Random Forest model trained successfully


In [38]:
# Task 2 - Random Forest Predictions

y_pred_rf = random_forest_model.predict(X_test)

y_prob_rf = random_forest_model.predict_proba(X_test)

print("Random Forest predictions completed")

Random Forest predictions completed


In [39]:
# Task 2 - Random Forest Evaluation

rf_accuracy = accuracy_score(y_test, y_pred_rf)

rf_f1 = f1_score(
    y_test,
    y_pred_rf,
    average="weighted"
)

rf_roc_auc = roc_auc_score(
    y_test,
    y_prob_rf,
    multi_class="ovr"
)

print("Random Forest Accuracy:", rf_accuracy)
print("Random Forest F1 Score:", rf_f1)
print("Random Forest ROC AUC:", rf_roc_auc)

Random Forest Accuracy: 0.5876027830487034
Random Forest F1 Score: 0.5804787550964269
Random Forest ROC AUC: 0.5775758206607389


In [40]:
# Task 3 - Check Player Data

print(round_stats.columns.tolist())

NameError: name 'round_stats' is not defined

In [41]:

# Task 3 - Load Player Round Stats

round_stats = pd.read_csv(
    "afl_players_round_by_round_stats_raw - afl_players_round_by_round_stats_raw.csv.csv"
)

print("Rows:", len(round_stats))
print(round_stats.columns.tolist())


FileNotFoundError: [Errno 2] No such file or directory: 'afl_players_round_by_round_stats_raw - afl_players_round_by_round_stats_raw.csv.csv'

In [42]:
# Task 3 - Check Files in Current Folder

import os

print(os.listdir())

['afl_feature_table_v1.csv', 'team_matches_home_away_raw - team_matches_home_away_raw.csv.csv', 'win_prediction_model.ipynb']


In [43]:
# Task 3 - Load Player Stats

round_stats = pd.read_csv(
    "afl_players_round_by_round_stats_raw - afl_players_round_by_round_stats_raw.csv.csv"
)

print("Rows:", len(round_stats))
print("Columns:")
print(round_stats.columns.tolist())

Rows: 274089
Columns:
['id', 'team', 'year', 'career_game_count', 'opponent', 'round', 'result', 'jersey_num', 'kicks', 'marks', 'handballs', 'disposals', 'goals', 'behinds', 'hit_outs', 'tackles', 'rebound_50s', 'inside_50s', 'clearances', 'clangers', 'free_kicks_for', 'free_kicks_against', 'brownlow_votes', 'contested_possessions', 'uncontested_possessions', 'contested_marks', 'marks_inside_50', 'one_percenters', 'bounces', 'goal_assist', 'percentage_of_game_played', 'player_id', 'match_date', 'fantasy_points', 'score', 'margin']


In [44]:
# Task 3 - Check Player Stats

print(round_stats[["player_id", "team", "year", "round", "fantasy_points"]].head())
print()
print(round_stats["fantasy_points"].describe())

   player_id              team  year round  fantasy_points
0      45552    Hawthorn Hawks  1994    21              36
1      44356      Geelong Cats  2024     1              23
2      45955  Essendon Bombers  1999    10              67
3      45656  Western Bulldogs  1994    21              81
4      45929   Richmond Tigers  1997    10              32

count    274089.000000
mean         65.244187
std          28.110251
min         -10.000000
25%          45.000000
50%          64.000000
75%          84.000000
max         210.000000
Name: fantasy_points, dtype: float64


In [45]:
# Task 3 - Top Player Baseline

baseline_prediction = round_stats["fantasy_points"].mean()

print("Baseline Prediction:", baseline_prediction)

Baseline Prediction: 65.24418710710755


In [46]:
# Task 3 - Baseline MAE and RMSE

from sklearn.metrics import mean_absolute_error, mean_squared_error

baseline_predictions = np.full(
    len(round_stats),
    baseline_prediction
)

baseline_mae = mean_absolute_error(
    round_stats["fantasy_points"],
    baseline_predictions
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        round_stats["fantasy_points"],
        baseline_predictions
    )
)

print("Baseline MAE:", baseline_mae)
print("Baseline RMSE:", baseline_rmse)

Baseline MAE: 22.54007275709381
Baseline RMSE: 28.110199559783368


In [47]:
# Task 3 - Prepare Player Features and Target

X_player = round_stats[
    [
        "team",
        "opponent",
        "kicks",
        "marks",
        "handballs",
        "disposals",
        "goals",
        "tackles"
    ]
]

y_player = round_stats["fantasy_points"]

print("Features shape:", X_player.shape)
print("Target shape:", y_player.shape)

Features shape: (274089, 8)
Target shape: (274089,)


In [48]:
# Task 3 - Split Player Data

X_train_player, X_test_player, y_train_player, y_test_player = train_test_split(
    X_player,
    y_player,
    test_size=0.20,
    random_state=42
)

print("Training rows:", len(X_train_player))
print("Testing rows:", len(X_test_player))

Training rows: 219271
Testing rows: 54818


In [49]:
# Task 3 - Create Player Preprocessing

categorical_features_player = [
    "team",
    "opponent"
]

numerical_features_player = [
    "kicks",
    "marks",
    "handballs",
    "disposals",
    "goals",
    "tackles"
]

categorical_pipeline_player = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

numerical_pipeline_player = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

player_preprocessor = ColumnTransformer(
    transformers=[
        ("cat", categorical_pipeline_player, categorical_features_player),
        ("num", numerical_pipeline_player, numerical_features_player)
    ]
)

print("Player preprocessing is ready")

Player preprocessing is ready


In [53]:
# Task 3 - Train Linear Regression Model

from sklearn.linear_model import LinearRegression

player_model = Pipeline(
    steps=[
        ("preprocessor", player_preprocessor),
        ("model", LinearRegression())
    ]
)

player_model.fit(X_train_player, y_train_player)

print("Linear Regression model trained successfully")

Linear Regression model trained successfully


In [54]:
# Task 3 - Player Predictions

y_pred_player = player_model.predict(X_test_player)

print("Player predictions completed")

Player predictions completed


In [55]:
# Task 3 - Model Evaluation

player_mae = mean_absolute_error(
    y_test_player,
    y_pred_player
)

player_rmse = np.sqrt(
    mean_squared_error(
        y_test_player,
        y_pred_player
    )
)

print("Player Model MAE:", player_mae)
print("Player Model RMSE:", player_rmse)

Player Model MAE: 4.650625115325679
Player Model RMSE: 7.823969458390041


In [56]:
# Task 3 - Top 5 Hit Rate

actual_top_5 = y_test_player.nlargest(5).index
predicted_top_5 = pd.Series(y_pred_player, index=y_test_player.index).nlargest(5).index

hits = 0

for i in predicted_top_5:
    if i in actual_top_5:
        hits = hits + 1

top_5_hit_rate = hits / 5

print("Top 5 Hits:", hits)
print("Top 5 Hit Rate:", top_5_hit_rate)

Top 5 Hits: 4
Top 5 Hit Rate: 0.8


In [57]:
# Task 3 - Compare Baseline and Model

comparison = pd.DataFrame({
    "Metric": ["MAE", "RMSE", "Top 5 Hit Rate"],
    "Baseline": [
        baseline_mae,
        baseline_rmse,
        "-"
    ],
    "Linear Regression": [
        player_mae,
        player_rmse,
        top_5_hit_rate
    ]
})

comparison

,Metric,Baseline,Linear Regression
0,MAE,22.540073,4.650625
1,RMSE,28.1102,7.823969
2,Top 5 Hit Rate,-,0.800000


In [58]:
# Task 4 - Check Player Predictions

print("Actual Fantasy Points:")
print(y_test_player.head())

print()
print("Predicted Fantasy Points:")
print(y_pred_player[:5])

Actual Fantasy Points:
242184    44
171587    81
158858    74
168730    75
39511     57
Name: fantasy_points, dtype: int64

Predicted Fantasy Points:
[50.95824622 84.39679631 72.82839555 72.73564679 56.21214974]


In [59]:
# Task 5 - Match Winner Prediction Function

def predict_match_winner(team_name, opponent, venue, win_streak, last_5_avg_score, head_to_head_wins):
    
    data = pd.DataFrame({
        "team_name": [team_name],
        "opponent": [opponent],
        "venue": [venue],
        "win_streak": [win_streak],
        "last_5_avg_score": [last_5_avg_score],
        "head_to_head_wins": [head_to_head_wins]
    })
    
    prediction = logistic_model.predict(data)
    
    return prediction[0]

In [60]:
# Task 5 - Test Match Winner Function

prediction = predict_match_winner(
    "adelaide crows",
    "hawthorn hawks",
    "Adelaide Oval",
    2,
    85,
    10
)

print("Predicted Match Winner:", prediction)

Predicted Match Winner: Home Win


In [61]:
# Task 5 - Top Player Prediction Function

def predict_player_points(team, opponent, kicks, marks, handballs, disposals, goals, tackles):
    
    data = pd.DataFrame({
        "team": [team],
        "opponent": [opponent],
        "kicks": [kicks],
        "marks": [marks],
        "handballs": [handballs],
        "disposals": [disposals],
        "goals": [goals],
        "tackles": [tackles]
    })
    
    prediction = player_model.predict(data)
    
    return prediction[0]

In [62]:
# Task 5 - Test Top Player Function

sample = X_test_player.iloc[0]

prediction = predict_player_points(
    sample["team"],
    sample["opponent"],
    sample["kicks"],
    sample["marks"],
    sample["handballs"],
    sample["disposals"],
    sample["goals"],
    sample["tackles"]
)

print("Predicted Fantasy Points:", prediction)

Predicted Fantasy Points: 50.95824622389422


In [63]:
# Task 5 - Save Models

import joblib

joblib.dump(logistic_model, "match_winner_model.pkl")
joblib.dump(player_model, "top_player_model.pkl")

print("Models saved successfully")

Models saved successfully
